In [2]:
#1. 数据加载与结构解析
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score)
import shap
import warnings
warnings.filterwarnings('ignore')

# 加载数据
df = pd.read_excel('AOactivity.xlsx')
print("数据集基本信息:", df.shape)

# 列名适配
compound_id_col = "chemical ID" if "chemical ID" in df.columns else "compound_id"
ao_label_col = "AO1065"

# 因果链节点（已修复，无标签）
causal_chain_nodes = [
    "MIE2293", "KE1076", "KE2294", "KE2251",
    "MIE1046", "KE1047", "KE1050", "KE1972", "KE1973",
    "MIE1065", "KE1985", "KE2137", "KE2402",
    "MIE2155", "MIE2250",
    "KE530", "KE531", "KE1695", "KE2303",
    "KE1075"
]

# 筛选特征
feature_cols = [col for col in causal_chain_nodes if col in df.columns and col != ao_label_col]
print(f"特征数: {len(feature_cols)}")
df[feature_cols + [ao_label_col]] = df[feature_cols + [ao_label_col]].fillna(0)

# ============== 纯 MLP 数据准备 ==============
X = df[feature_cols].values
y = df[ao_label_col].values
compound_ids = df[compound_id_col].values

# 划分数据集（和原图版本完全一致）
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42
)

# 转为张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# 数据加载器
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False)

# ===== 纯全连接神经网络 MLP =====
class CausalChainMLP(torch.nn.Module):
    def __init__(self, hidden_dim=32, dropout=0.2):
        super().__init__()
        # 完全保留你原来的参数、层数、维度
        self.fc1 = torch.nn.Linear(len(feature_cols), hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, 16)
        self.fc3 = torch.nn.Linear(16, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x):
        # 标准MLP前向传播，无任何图操作
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# ===== 优化器 + 损失 + 学习率调度器（完全不变）=====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CausalChainMLP(hidden_dim=32, dropout=0.2).to(device)

pos_weight = torch.tensor([1.8], device=device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# 训练函数（适配MLP，极简无图）
def train_epoch():
    model.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        out = model(x_batch)
        loss = criterion(out, y_batch.unsqueeze(1))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x_batch.size(0)
    return total_loss / len(train_dataset)

# 评估函数（完全不变）
@torch.no_grad()
def test_eval():
    model.eval()
    y_true, y_prob = [], []
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        logits = model(x_batch)
        prob = torch.sigmoid(logits).cpu().numpy()
        y_true.extend(y_batch.cpu().numpy())
        y_prob.extend(prob)
    
    y_true = np.array(y_true).flatten()
    y_prob = np.array(y_prob).flatten()

    best_f1 = 0
    best_t = 0.3
    for t in np.linspace(0.2, 0.5, 41):
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    y_pred = (y_prob >= best_t).astype(int)
    
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    return acc, auc, best_f1

# ========== 训练循环（完全不变）==========
print("\n开始训练...")
best_auc = 0
best_f1 = 0
patience = 0
early_stop_patience = 30

for epoch in range(1, 151):
    loss = train_epoch()
    acc, auc, f1 = test_eval()

    if auc > best_auc:
        best_auc = auc
        best_f1 = f1
        torch.save(model.state_dict(), "best_mlp.pth")
        patience = 0
    else:
        patience += 1

    if patience >= early_stop_patience:
        print(f"⏹️ 早停触发 Epoch {epoch}")
        break

    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Acc {acc:.4f} | AUC {auc:.4f} | F1 {f1:.4f}")

model.load_state_dict(torch.load("best_mlp.pth"))
final_acc, final_auc, final_f1 = test_eval()
print("\n✅ 最终最优结果：")
print(f"Acc: {final_acc:.4f} | AUC: {final_auc:.4f} | F1分数: {final_f1:.4f}")



数据集基本信息: (4282, 24)
特征数: 20

开始训练...
Epoch 010 | Loss 0.5813 | Acc 0.8028 | AUC 0.7694 | F1 0.5185
Epoch 020 | Loss 0.5593 | Acc 0.8121 | AUC 0.7701 | F1 0.5194
Epoch 030 | Loss 0.5522 | Acc 0.8098 | AUC 0.7683 | F1 0.5192
Epoch 040 | Loss 0.5548 | Acc 0.8203 | AUC 0.7682 | F1 0.5417
⏹️ 早停触发 Epoch 44

✅ 最终最优结果：
Acc: 0.8063 | AUC: 0.7731 | F1分数: 0.5257
